In [1]:
import gc
from pathlib import Path

import lightgbm as lgb
import numpy as np
import polars as pl
from sklearn.preprocessing import TargetEncoder

# Configure Polars print outs
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)

polars.config.Config

In [2]:
RAW_INPUT_DATA_DIR = Path("/kaggle/input/competitions/m5-forecasting-accuracy")
FEATURES_DATA_PATH = Path("/kaggle/input/datasets/lmalms/m5-acc-2-0/m5-acc-features-df.pq")
SUBMISSION_DIR = Path("/kaggle/working/")

In [3]:
# Data constants
LABEL_COL_NAME = "sales"
MAX_TRAIN_INDEX = 1941
VALIDATION_FORECAST_HORIZON = 28
VALID_START_INDEX = MAX_TRAIN_INDEX - VALIDATION_FORECAST_HORIZON + 1
SUBMISSION_FORECAST_HORIZON = 28

# Only the most recent MAX_TRAIN_DAYS days are used for training + validation.
# Lower this to reduce RAM usage (None = use everything).
MAX_TRAIN_DAYS: int | None = None

# Feature engineering constants. These must match what the feature engineering
# notebook (1.1) used to build the features parquet.
SALES_LAGS = [1, 2, 3, 4, 7, 14]
ROLLING_SALES_WINDOWS = [2, 3, 4, 7, 14]
PRICE_LAGS = [1, 2, 3, 4, 7]
ROLLING_PRICE_WINDOWS = [2, 3, 4, 7, 14]

CYCLICAL_CALENDAR_FEATURE_SUFFIXES = ["wday", "month"]

N_EVENTS = 2
MAX_DAYS_UNTIL_EVENT = 7
MAX_DAYS_SINCE_EVENT = 7

SNAP_STATES = ["CA", "TX", "WI"]
ELECTION_COLS = [f"snap_{s}" for s in SNAP_STATES]
MAX_DAYS_UNTIL_ELECTION = 7
MAX_DAYS_SINCE_ELECTION = 7

### Feature configuration

In [4]:
# Feature configuration
ALL_FEATURE_GROUPS: dict[str, list[str]] = {
    "id_categorical": ["item_id", "dept_id", "cat_id", "store_id", "state_id"],

    # Sales features
    "sales_lags": [f"sales_lagged_{lag}" for lag in SALES_LAGS],
    "sales_rolling_mean": [f"sales_rolling_mean_{w}" for w in ROLLING_SALES_WINDOWS],
    
    # Price features
    "sell_price": ["sell_price"],
    "price_lags": [f"sell_price_lagged_{lag}" for lag in PRICE_LAGS],
    "price_rolling_mean": [f"sell_price_rolling_mean_{w}" for w in ROLLING_PRICE_WINDOWS],
    "price_rolling_std": [f"sell_price_rolling_std_{w}" for w in ROLLING_PRICE_WINDOWS],
    
    # Calendar features
    "calendar_cyclical": [
        f"{trig}_{suffix}"
        for suffix in CYCLICAL_CALENDAR_FEATURE_SUFFIXES
        for trig in ("sin", "cos")
    ],
    "calendar_raw": ["wday", "month", "year"],
    "is_weekend": ["is_weekend"],

    # Event features
    "event_categorical": [
        f"{prefix}_{i}"
        for prefix in ("event_name", "event_type", "last_event_type", "next_event_type")
        for i in range(1, N_EVENTS + 1)
    ],
    "event_timing": [
        f"days_{direction}_event_{i}"
        for direction in ("since", "until")
        for i in range(1, N_EVENTS + 1)
    ],

    # Election features
    "snap": list(ELECTION_COLS),
    "election_timing": [
        f"days_{direction}_{e}" 
        for direction in ("since", "until") 
        for e in ELECTION_COLS
    ],
}
CATEGORICAL_FEATURE_GROUPS = {"id_categorical", "event_categorical"}
TARGET_ENCODER_FEATURE_COLS = {"item_id"}

# ---- Configure features ----
# Add / remove groups to include / drop all of its features from training and inference.
FEATURE_GROUPS_TO_INCLUDE = [
    "id_categorical",
    "event_categorical",
    "sell_price",
    "sales_lags",
    "sales_rolling_mean",
    "price_lags",
    "price_rolling_mean",
    "price_rolling_std",
    "calendar_cyclical",
    # "calendar_raw",
    "is_weekend",
    "event_timing",
    # "snap",
    # "election_timing",
]

# Add / remove individual features
EXTRA_FEATURES_TO_INCLUDE: list[str] = []
FEATURES_TO_EXCLUDE: list[str] = []
# ------------------------

# Construct features
_categorical_candidates = {c for g in CATEGORICAL_FEATURE_GROUPS for c in ALL_FEATURE_GROUPS[g]}
FEATURE_COLS = list(
    dict.fromkeys(
        c
        for c in [*(c for g in FEATURE_GROUPS_TO_INCLUDE for c in ALL_FEATURE_GROUPS[g]), *EXTRA_FEATURES_TO_INCLUDE]
        if c not in FEATURES_TO_EXCLUDE
    )
)
CATEGORICAL_FEATURE_COLS = [c for c in FEATURE_COLS if c in _categorical_candidates]

print(f"{len(FEATURE_COLS)} features ({len(CATEGORICAL_FEATURE_COLS)} categorical)")

49 features (13 categorical)


### Model configuration

In [5]:
# LightGBM config
NUM_BOOST_ROUND = 1000
EARLY_STOPPING_ROUNDS = 10

LGB_PARAMS = {
    "objective": "poisson",
    "metric": "rmse",
    "learning_rate": 2e-03,         
    "num_leaves": 63,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l2": 1.0,
    "max_bin": 255,
    "cat_smooth": 20,
    "force_row_wise": True,        
    "seed": 0,
    "num_threads": -1,
    "verbosity": -1,
}

### Load data

In [6]:
DATA_COLS = list(dict.fromkeys(["id", "d_index", LABEL_COL_NAME, *FEATURE_COLS]))
MIN_TRAIN_INDEX = 0 if MAX_TRAIN_DAYS is None else MAX_TRAIN_INDEX - MAX_TRAIN_DAYS

full_df = (
    pl.scan_parquet(FEATURES_DATA_PATH)
    .select(DATA_COLS)
    .filter(pl.col("d_index") > MIN_TRAIN_INDEX)
    .with_columns(pl.col(pl.Float64).cast(pl.Float32))
    .collect()
)
print(f"{full_df.estimated_size('gb'):.2f} GB")
full_df.shape

8.22 GB


(46881677, 52)

In [7]:
# For submission, features are constructed iteratively. Keep only the final slice of
# the sales / price history needed to construct features for the first forecast day,
# so the full feature df isn't needed at inference.
max_lookback = max(SALES_LAGS + ROLLING_SALES_WINDOWS + PRICE_LAGS + ROLLING_PRICE_WINDOWS)

history_df = (
    pl.scan_parquet(FEATURES_DATA_PATH)
    .select(["id", "d_index", "sales", "sell_price"])
    .filter(pl.col("d_index") > MAX_TRAIN_INDEX - max_lookback)
    .with_columns(pl.col("sell_price").cast(pl.Float32))
    .sort(["id", "d_index"])
    .collect()
)
sales_history_df = history_df.select(["id", "d_index", "sales"])
price_history_df = history_df.select(["id", "d_index", "sell_price"])
del history_df
gc.collect()

sales_history_df.head(max_lookback + 2)

id,d_index,sales
cat,i16,i16
"""FOODS_1_001_CA_1_evaluation""",1928,0
"""FOODS_1_001_CA_1_evaluation""",1929,2
"""FOODS_1_001_CA_1_evaluation""",1930,0
"""FOODS_1_001_CA_1_evaluation""",1931,0
"""FOODS_1_001_CA_1_evaluation""",1932,2
"""FOODS_1_001_CA_1_evaluation""",1933,3
"""FOODS_1_001_CA_1_evaluation""",1934,1
"""FOODS_1_001_CA_1_evaluation""",1935,0
"""FOODS_1_001_CA_1_evaluation""",1936,0


## Prepare data

In [8]:
TARGET_ENCODER = TargetEncoder(smooth="auto", target_type="continuous")

# Construct category mapping from training data only for any categories
# that will not be target encoded.
CATEGORY_DTYPES = {
    col: pl.Enum(
        full_df.filter(pl.col("d_index") < VALID_START_INDEX)[col]
        .cast(pl.String).drop_nulls().unique().sort()
    )
    for col in CATEGORICAL_FEATURE_COLS
    if col not in TARGET_ENCODER_FEATURE_COLS
}


def to_float32_matrix(df: pl.DataFrame, is_train: bool) -> np.ndarray:
    """
    Feature matrix (float32, NaN = missing) in FEATURE_COLS order.
    Categoricals become integer codes (LightGBM treats them as categories through `categorical_feature`).
    """
    X = np.empty((df.height, len(FEATURE_COLS)), dtype=np.float32)

    # Prepare target encodings
    target_encoder_cols = [c for c in FEATURE_COLS if c in TARGET_ENCODER_FEATURE_COLS]
    if target_encoder_cols:
        if is_train:
            X_target_encoded = TARGET_ENCODER.fit_transform(
                df[target_encoder_cols].to_numpy(),
                df[LABEL_COL_NAME].to_numpy().astype(np.float32)
            )
        else:
            X_target_encoded = TARGET_ENCODER.transform(
                df[target_encoder_cols].to_numpy(),
            )
    
    for j, col in enumerate(FEATURE_COLS):
        s = df.get_column(col)
        if col in CATEGORY_DTYPES:
            s = s.cast(pl.String).cast(CATEGORY_DTYPES[col], strict=False).to_physical()
            v = s.cast(pl.Float32).to_numpy()
        elif col in target_encoder_cols:
            v = X_target_encoded[:, target_encoder_cols.index(col)]
        else:
            v = s.cast(pl.Float32).to_numpy()
        X[:, j] = v
    return X


In [9]:
# RAM: build one split at a time and delete the polars frames as soon as possible.
is_valid = pl.col("d_index") >= VALID_START_INDEX

train_df = full_df.filter(~is_valid)
X_train = to_float32_matrix(train_df, is_train=True)
y_train = train_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
del train_df
gc.collect()

valid_df = full_df.filter(is_valid)
X_valid = to_float32_matrix(valid_df, is_train=False)
y_valid = valid_df[LABEL_COL_NAME].to_numpy().astype(np.float32)
del valid_df, full_df
gc.collect()
print(X_train.shape, X_valid.shape)

train_set = lgb.Dataset(
    X_train,
    label=y_train,
    feature_name=FEATURE_COLS,
    categorical_feature=CATEGORICAL_FEATURE_COLS,
    params=LGB_PARAMS,
    free_raw_data=True,
)
valid_set = lgb.Dataset(
    X_valid,
    label=y_valid,
    reference=train_set,
    feature_name=FEATURE_COLS,
    categorical_feature=CATEGORICAL_FEATURE_COLS,
    params=LGB_PARAMS,
    free_raw_data=True,
)

# RAM: binning the data into LightGBM's compact histogram format happens at construct();
# afterwards the raw float32 matrices are no longer needed.
train_set.construct()
valid_set.construct()
del X_train, X_valid, y_train, y_valid
gc.collect()

(46027957, 49) (853720, 49)


39

## Model

In [10]:
evals_result: dict = {}
model = lgb.train(
    LGB_PARAMS,
    train_set,
    num_boost_round=NUM_BOOST_ROUND,
    valid_sets=[valid_set],
    valid_names=["valid"],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=10),
        lgb.record_evaluation(evals_result),
    ],
)
print(f"Best iteration: {model.best_iteration}, "
      f"valid RMSE: {model.best_score['valid']['rmse']:.4f}")

Training until validation scores don't improve for 10 rounds
[10]	valid's rmse: 3.61582
[20]	valid's rmse: 3.59128
[30]	valid's rmse: 3.56696
[40]	valid's rmse: 3.54308
[50]	valid's rmse: 3.51951
[60]	valid's rmse: 3.49618
[70]	valid's rmse: 3.47314
[80]	valid's rmse: 3.45045
[90]	valid's rmse: 3.42799
[100]	valid's rmse: 3.40578
Did not meet early stopping. Best iteration is:
[100]	valid's rmse: 3.40578
Best iteration: 100, valid RMSE: 3.4058


In [11]:
# RAM: the datasets are no longer needed
del train_set, valid_set
gc.collect()

242

In [12]:
(
    pl.DataFrame({
        "feature": model.feature_name(),
        "gain": model.feature_importance(importance_type="gain"),
    })
    .sort("gain", descending=True)
    .with_columns(gain_share=pl.col("gain") / pl.col("gain").sum())
)

feature,gain,gain_share
str,f64,f64
"""sales_rolling_mean_7""",5.4697e9,0.649109
"""sales_rolling_mean_14""",2.2264e9,0.264211
"""sales_rolling_mean_3""",2.3011e8,0.027307
"""sales_lagged_1""",1.8495e8,0.021949
"""sales_rolling_mean_4""",1.3543e8,0.016072
"""sin_wday""",5.7223e7,0.006791
"""sales_rolling_mean_2""",5.4138e7,0.006425
"""item_id""",1.9169e7,0.002275
"""sales_lagged_14""",1.4463e7,0.001716


## Submission

In [13]:
# Load data needed for inference

# Calendar data
calendar_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    **{date_type: pl.Int16 for date_type in ["wday", "month", "year"]},
    **{f"snap_{state}": pl.Int16 for state in SNAP_STATES},
}
raw_calendar_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "calendar.csv",
    try_parse_dates=True,
    schema_overrides=calendar_schema_overrides,
)

# Sell price data
sell_price_schema_overrides = {
    "wm_yr_wk": pl.Int16,
    "sell_price": pl.Float32
}
raw_sell_price_df = pl.read_csv(
    RAW_INPUT_DATA_DIR / "sell_prices.csv",
    schema_overrides=sell_price_schema_overrides,
)

# Submission
sample_submission_df = pl.read_csv(RAW_INPUT_DATA_DIR / "sample_submission.csv")

In [14]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

event_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "event_name_1", "event_type_1", "event_name_2", "event_type_2"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(f"event_name_{i}").is_not_null())
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"event_index_{i}")
        for i in range(1, N_EVENTS + 1)
    )
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"event_index_{i}").forward_fill())
            .alias(f"days_since_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").forward_fill().alias(f"last_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            (pl.col(f"event_index_{i}").backward_fill() - pl.col("d_index"))
            .alias(f"days_until_event_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.col(f"event_type_{i}").backward_fill().alias(f"next_event_type_{i}")
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .with_columns(
        # Clip days since / until events (and the associated event types)
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"days_since_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_since_event_{i}") <= MAX_DAYS_SINCE_EVENT)
            .then(pl.col(f"last_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"days_until_event_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
        *[
            pl.when(pl.col(f"days_until_event_{i}") <= MAX_DAYS_UNTIL_EVENT)
            .then(pl.col(f"next_event_type_{i}"))
            .otherwise(None)
            for i in range(1, N_EVENTS + 1)
        ],
    )
    .drop([f"event_index_{i}" for i in range(1, N_EVENTS + 1)])
)

event_features_df.head()

wm_yr_wk,d,date,event_name_1,event_type_1,event_name_2,event_type_2,d_index,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2
i16,str,date,str,str,str,str,i16,i16,i16,str,str,i16,i16,str,str
11101,"""d_1""",2011-01-29,null,null,null,null,1,null,null,null,null,null,null,null,null
11101,"""d_2""",2011-01-30,null,null,null,null,2,null,null,null,null,7,null,"""Sporting""",null
11101,"""d_3""",2011-01-31,null,null,null,null,3,null,null,null,null,6,null,"""Sporting""",null
11101,"""d_4""",2011-02-01,null,null,null,null,4,null,null,null,null,5,null,"""Sporting""",null
11101,"""d_5""",2011-02-02,null,null,null,null,5,null,null,null,null,4,null,"""Sporting""",null


In [15]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

calendar_features_df = (
    raw_calendar_df
    .select(["wm_yr_wk", "d", "date", "wday", "month", "year"])
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        sin_wday=(2 * np.pi * pl.col("wday") / 7).sin().cast(pl.Float32),
        cos_wday=(2 * np.pi * pl.col("wday") / 7).cos().cast(pl.Float32),
        sin_month=(2 * np.pi * pl.col("month") / 12).sin().cast(pl.Float32),
        cos_month=(2 * np.pi * pl.col("month") / 12).cos().cast(pl.Float32),
        is_weekend=(pl.col("wday") <= 2).cast(pl.Float32),
    )
)

calendar_features_df.head()

wm_yr_wk,d,date,wday,month,year,d_index,sin_wday,cos_wday,sin_month,cos_month,is_weekend
i16,str,date,i16,i16,i16,i16,f32,f32,f32,f32,f32
11101,"""d_1""",2011-01-29,1,1,2011,1,0.781832,0.62349,0.5,0.866025,1.0
11101,"""d_2""",2011-01-30,2,1,2011,2,0.974928,-0.222521,0.5,0.866025,1.0
11101,"""d_3""",2011-01-31,3,1,2011,3,0.433884,-0.900969,0.5,0.866025,0.0
11101,"""d_4""",2011-02-01,4,2,2011,4,-0.433884,-0.900969,0.866025,0.5,0.0
11101,"""d_5""",2011-02-02,5,2,2011,5,-0.974928,-0.222521,0.866025,0.5,0.0


In [16]:
# Calendar / event / election features are recomputed for inference.
# Logic must match what the feature engineering notebook (1.1) used to build the features parquet.

election_features_df = (
    raw_calendar_df
    .with_columns(d_index=pl.col("d").str.strip_prefix("d_").cast(pl.Int16))
    .with_columns(
        pl.when(pl.col(election).eq(1))
        .then(pl.col("d_index"))
        .otherwise(None)
        .alias(f"{election}_index")
        for election in ELECTION_COLS
    )
    .select(["wm_yr_wk", "date", "d", "d_index"] + ELECTION_COLS + [f"{e}_index" for e in ELECTION_COLS])
    .with_columns(
        *[
            (pl.col("d_index") - pl.col(f"{e}_index").forward_fill()).alias(f"days_since_{e}")
            for e in ELECTION_COLS
        ],
        *[
            (pl.col(f"{e}_index").backward_fill() - pl.col("d_index")).alias(f"days_until_{e}")
            for e in ELECTION_COLS
        ],
    )
    .with_columns(
        *[
            pl.when(pl.col(f"days_since_{e}") <= MAX_DAYS_SINCE_ELECTION)
            .then(pl.col(f"days_since_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
        *[
            pl.when(pl.col(f"days_until_{e}") <= MAX_DAYS_UNTIL_ELECTION)
            .then(pl.col(f"days_until_{e}"))
            .otherwise(None)
            for e in ELECTION_COLS
        ],
    )
    .drop([f"{e}_index" for e in ELECTION_COLS])
)

election_features_df.head()

wm_yr_wk,date,d,d_index,snap_CA,snap_TX,snap_WI,days_since_snap_CA,days_since_snap_TX,days_since_snap_WI,days_until_snap_CA,days_until_snap_TX,days_until_snap_WI
i16,date,str,i16,i16,i16,i16,i16,i16,i16,i16,i16,i16
11101,2011-01-29,"""d_1""",1,0,0,0,null,null,null,3,3,4
11101,2011-01-30,"""d_2""",2,0,0,0,null,null,null,2,2,3
11101,2011-01-31,"""d_3""",3,0,0,0,null,null,null,1,1,2
11101,2011-02-01,"""d_4""",4,1,1,0,0,0,null,0,0,1
11101,2011-02-02,"""d_5""",5,1,0,1,0,1,0,0,1,0


In [17]:
# Merge calendar / events / election features once
# and keep only the forecast days (small).

forecast_calendar_df = (
    calendar_features_df.drop("d", "date")
    .join(event_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .join(election_features_df.drop("d", "date"), on=["wm_yr_wk", "d_index"], how="inner", validate="1:1")
    .filter(pl.col("d_index") > MAX_TRAIN_INDEX)
)
forecast_calendar_df.head()

wm_yr_wk,wday,month,year,d_index,sin_wday,cos_wday,sin_month,cos_month,is_weekend,event_name_1,event_type_1,event_name_2,event_type_2,days_since_event_1,days_since_event_2,last_event_type_1,last_event_type_2,days_until_event_1,days_until_event_2,next_event_type_1,next_event_type_2,snap_CA,snap_TX,snap_WI,days_since_snap_CA,days_since_snap_TX,days_since_snap_WI,days_until_snap_CA,days_until_snap_TX,days_until_snap_WI
i16,i16,i16,i16,i16,f32,f32,f32,f32,f32,str,str,str,str,i16,i16,str,str,i16,i16,str,str,i16,i16,i16,i16,i16,i16,i16,i16,i16
11617,3,5,2016,1942,0.433884,-0.900969,0.5,-0.866025,0.0,null,null,null,null,null,null,null,null,7,null,"""National""",null,0,0,0,null,null,null,null,null,null
11617,4,5,2016,1943,-0.433884,-0.900969,0.5,-0.866025,0.0,null,null,null,null,null,null,null,null,6,null,"""National""",null,0,0,0,null,null,null,null,null,null
11617,5,5,2016,1944,-0.974928,-0.222521,0.5,-0.866025,0.0,null,null,null,null,null,null,null,null,5,null,"""National""",null,0,0,0,null,null,null,7,7,null
11617,6,5,2016,1945,-0.781832,0.62349,0.5,-0.866025,0.0,null,null,null,null,null,null,null,null,4,null,"""National""",null,0,0,0,null,null,null,6,6,7
11617,7,5,2016,1946,-2.4493e-16,1.0,0.5,-0.866025,0.0,null,null,null,null,null,null,null,null,3,null,"""National""",null,0,0,0,null,null,null,5,5,6


In [18]:
# Forecast for submission

SUBMISSION_F_COLS = [f"F{i}" for i in range(1, SUBMISSION_FORECAST_HORIZON + 1)]

_SERIES_ID_RE = (
    r"^(?P<cat>[A-Z]+)_(?P<dept>\d+)_(?P<item>\d+)_(?P<state>[A-Z]+)_(?P<store>\d+)"
    r"_(?:validation|evaluation)$"
)

FORECAST_IDS = (
    sample_submission_df
    .filter(sample_submission_df["id"].str.ends_with("_evaluation"))
    ["id"]
    .unique()
    .to_list()
)

# Recursive forecasting: iterate over forecast days, predict, and feed the predictions
# back in as sales history for the features of the next day.

all_forecast_dfs: list[pl.DataFrame] = []

for f_index in range(1, SUBMISSION_FORECAST_HORIZON + 1):
    d_index = MAX_TRAIN_INDEX + f_index

    # Construct test_df with identifier columns
    test_df = (
        pl.DataFrame({"id": FORECAST_IDS})
        .with_columns(parsed_id=pl.col("id").str.extract_groups(_SERIES_ID_RE))
        .with_columns(
            item_id=pl.format(
                "{}_{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
                pl.col("parsed_id").struct.field("item"),
            ),
            dept_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("cat"),
                pl.col("parsed_id").struct.field("dept"),
            ),
            cat_id=pl.col("parsed_id").struct.field("cat"),
            store_id=pl.format(
                "{}_{}",
                pl.col("parsed_id").struct.field("state"),
                pl.col("parsed_id").struct.field("store"),
            ),
            state_id=pl.col("parsed_id").struct.field("state"),
        )
        .drop("parsed_id")
        .with_columns(
            id=pl.col("id").cast(pl.Categorical),
            d_index=pl.lit(d_index).cast(pl.Int16),
            sales=pl.lit(None).cast(pl.Int16),
        )
        # Prices and calendar features for the current d_index are known in advance
        .join(
            other=forecast_calendar_df.select(["d_index", "wm_yr_wk"]),
            on="d_index", how="left", validate="m:1",
        )
        .join(
            other=raw_sell_price_df,
            on=["item_id", "store_id", "wm_yr_wk"], how="left", validate="m:1",
        )
        .drop("wm_yr_wk")
    )

    # Lagged / rolling sales features
    previous_sales_df = (
        pl.concat(
            [
                sales_history_df,
                *[
                    df.select(["id", "d_index", "sales"])
                    .with_columns(id=pl.col("id").cast(pl.Categorical))
                    for df in all_forecast_dfs
                ],
                test_df.select(["id", "d_index", "sales"]),
            ]
        )
        .sort(by=["id", "d_index"])
    )

    test_sales_features_df = (
        previous_sales_df
        .with_columns(
            [
                pl.col("sales").shift(lag).over("id", order_by="d_index")
                .alias(f"sales_lagged_{lag}")
                for lag in SALES_LAGS
            ]
            + [
                pl.col("sales").shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sales_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_SALES_WINDOWS
            ]
        )
        .drop("sales")
        .filter(pl.col("d_index") == d_index)
    )

    # Lagged / rolling price features
    price_history_df = (
        pl.concat([
            price_history_df,
            test_df.select(["id", "d_index", "sell_price"])
        ])
        .sort(by=["id", "d_index"])
    )

    test_price_features_df = (
        price_history_df
        .with_columns(
            [
                pl.col("sell_price").shift(lag).over("id", order_by="d_index")
                .alias(f"sell_price_lagged_{lag}")
                for lag in PRICE_LAGS
            ]
            + [
                pl.col("sell_price").shift(1)
                .rolling_mean(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_mean_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
            + [
                pl.col("sell_price").shift(1)
                .rolling_std(window_size=window, min_samples=window)
                .over("id", order_by="d_index")
                .alias(f"sell_price_rolling_std_{window}")
                .cast(pl.Float32)
                for window in ROLLING_PRICE_WINDOWS
            ]
        )
        .drop("sell_price")
        .filter(pl.col("d_index") == d_index)
    )
    # Keep the price history bounded
    price_history_df = price_history_df.filter(pl.col("d_index") > d_index - max_lookback)

    # Join id, sales, price and calendar features
    test_df = (
        test_df.drop("sales")
        .join(test_sales_features_df, on=["id", "d_index"], how="inner", validate="1:1")
        .join(test_price_features_df, on=["id", "d_index"], how="inner", validate="1:1")
        .join(forecast_calendar_df, on="d_index", how="inner", validate="m:1")
        # Series without a price at this timestamp aren't sold: no forecast
        .filter(pl.col("sell_price").is_not_null())
    )

    # Make predictions. The poisson objective makes `predict` return the expected count
    # (the inverse link is applied). Use the best iteration found by early stopping.
    y_hat = model.predict(to_float32_matrix(test_df, is_train=False), num_iteration=model.best_iteration)

    # Predictions are aligned with the rows of test_df. Sales can't be negative, and are
    # rounded to integers so they can be fed back in as lagged sales features.
    y_hat_df = (
        pl.DataFrame({"id": test_df["id"].cast(pl.String), "sales": y_hat})
        .with_columns(
            sales=pl.col("sales").clip(lower_bound=0).round().cast(pl.Int16),
            d_index=pl.lit(d_index).cast(pl.Int16),
            F=pl.format("F{}", f_index),
        )
    )
    all_forecast_dfs.append(y_hat_df)

In [19]:
final_forecast_df = (
    pl.concat(all_forecast_dfs)
    .with_columns(sales=pl.col("sales").cast(pl.Int64))
    .pivot(on="F", index="id", values="sales")
)

FINAL_SUBMISSION_DF = (
    sample_submission_df
    .join(final_forecast_df, on="id", how="left", suffix="_model", validate="1:1")
    .with_columns(
        pl.coalesce(pl.col(f"{c}_model"), pl.col(c)).cast(pl.Int64).alias(c)
        for c in SUBMISSION_F_COLS
    )
    .select(sample_submission_df.columns)
)

FINAL_SUBMISSION_DF.write_csv(SUBMISSION_DIR / "submission.csv")